In [1]:
import pandas as pd
from pathlib import Path
from decimal import Decimal, InvalidOperation

FILE = Path(r"C:/Users/Carl/Desktop/hh_id_obs_merge.xlsx")

def clean_id(x):
    if pd.isna(x):
        return pd.NA

    s = str(x).strip()

    if s == "" or s.lower() == "nan":
        return pd.NA

    try:
        if "e" in s.lower():
            s = format(Decimal(s), "f")

        if s.endswith(".0"):
            s = s[:-2]

    except InvalidOperation:
        pass

    return s


left = pd.read_excel(FILE, sheet_name="Table1", dtype=str)
right = pd.read_excel(FILE, sheet_name="Sheet1 (2)", dtype=str)

left["Wave_clean"] = pd.to_numeric(left["Wave"], errors="coerce").astype("Int64")
left["HHID_clean"] = left["HHID"].map(clean_id)

right["Wave_clean"] = pd.to_numeric(right["wave"], errors="coerce").astype("Int64")
right["HHID_clean"] = right["hh_id_merge"].map(clean_id)

# Sheet1 (2) has season rows, so reduce to one row per Wave-HHID first
right_map = (
    right[["Wave_clean", "HHID_clean", "hh_id_obs"]]
    .drop_duplicates()
)

conflicts = (
    right_map
    .groupby(["Wave_clean", "HHID_clean"])["hh_id_obs"]
    .nunique(dropna=True)
    .reset_index(name="n_hh_id_obs")
    .query("n_hh_id_obs > 1")
)

print("Conflicting hh_id_obs mappings:", len(conflicts))
display(conflicts.head(20))

right_map = (
    right_map
    .drop_duplicates(["Wave_clean", "HHID_clean"])
)

merged = left.merge(
    right_map,
    on=["Wave_clean", "HHID_clean"],
    how="left",
    indicator=True
)

print(merged["_merge"].value_counts(dropna=False))

matched = merged[merged["_merge"].eq("both")].copy()
unmatched = merged[merged["_merge"].eq("left_only")].copy()

display(matched.head())
display(unmatched.head())

Conflicting hh_id_obs mappings: 0


,Wave_clean,HHID_clean,n_hh_id_obs


_merge
both          26072
left_only      5187
right_only        0
Name: count, dtype: int64


,Wave,HHID,Wave_clean,HHID_clean,hh_id_obs,_merge
30,1,1013000204,1,1013000204,7000045,both
31,1,1021000108,1,1021000108,7000052,both
32,1,1021000108,1,1021000108,7000052,both
33,1,1021000113,1,1021000113,7000056,both
34,1,1021000113,1,1021000113,7000056,both


,Wave,HHID,Wave_clean,HHID_clean,hh_id_obs,_merge
0,1,102100000000,1,102100000000,NaN,left_only
1,1,102100000000,1,102100000000,NaN,left_only
2,1,103300000000,1,103300000000,NaN,left_only
3,1,103300000000,1,103300000000,NaN,left_only
4,1,104300000000,1,104300000000,NaN,left_only


In [2]:
OUT = Path(r"C:/Users/Carl/Desktop")

matched.to_excel(OUT / "hh_id_obs_merge_matched_only.xlsx", index=False)
unmatched.to_excel(OUT / "hh_id_obs_merge_unmatched.xlsx", index=False)